# 05 — Approach C: Few-shot LLM with embedding retrieval

For each test record:
1. Encode it with the same bge-base model used in Approach B.
2. Retrieve the top-3 most similar `rule_based` train records (cosine over normalized embeddings).
3. Inject those 3 records as concrete few-shot examples into the LLM prompt.
4. Run a small LLM panel on both benchmarks: `rule_based` and `hand_labelled`.

Hypothesis: retrieval helps most on `app_specific`, where the correct label often depends on matching the feedback tags to the agent domain. The LLM still outputs only four real labels: `junk`, `service_feedback`, `config_feedback`, and `app_specific`.


In [ ]:
import sys, json
from pathlib import Path
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(AI_ROOT))

import numpy as np, pandas as pd
from tqdm.auto import tqdm
from shared.mongo_client import fetch_agents_by_keys
from shared.ollama_client import OllamaClient
from shared.prompts import SYSTEM_V4
from shared.context_builder import build_user_message, build_embedding_text
from shared.types import AgentMeta, FeedbackRecord
from shared.metrics import per_class_report, confusion_df, summarize_run

In [ ]:
SPLITS = AI_ROOT / 'data' / 'splits'
EMB_DIR = AI_ROOT / 'data' / 'embeddings'
TRAIN = pd.read_parquet(SPLITS / 'rule_based' / 'train.parquet')
TESTS = {
    'rule_based': pd.read_parquet(SPLITS / 'rule_based' / 'test.parquet'),
    'hand_labelled': pd.read_parquet(SPLITS / 'hand_labelled' / 'test.parquet'),
}

# Reuse cached embeddings from 04_approach_b.
X_train = np.load(EMB_DIR / 'rule_based_train_bge_base.npy')
X_test_by_bench = {
    bench: np.load(EMB_DIR / f'{bench}_test_bge_base.npy')
    for bench in TESTS
}
y_train = TRAIN['label'].tolist()
print('train emb:', X_train.shape)
for bench, x in X_test_by_bench.items():
    print(f'{bench:13s} test emb:', x.shape)

In [ ]:
# Cosine sim ≡ dot product when embeddings are L2-normalized (bge default).
K = 3

def retrieve_topk(X_test: np.ndarray) -> np.ndarray:
    sims = X_test @ X_train.T
    topk = np.argpartition(-sims, K, axis=1)[:, :K]
    # Sort the top-K per row by sim desc so the closest example comes first.
    for i in range(len(topk)):
        order = np.argsort(-sims[i, topk[i]])
        topk[i] = topk[i][order]
    return topk

topk_by_bench = {bench: retrieve_topk(x) for bench, x in X_test_by_bench.items()}
for bench, topk_idx in topk_by_bench.items():
    print(f'{bench} test[0] →', topk_idx[0], '(labels:', [y_train[j] for j in topk_idx[0]], ')')

In [ ]:
# Build agent context cache for train and both benchmark test rows.
all_keys = set()
for df in [TRAIN, *TESTS.values()]:
    for r in df.itertuples():
        all_keys.add((int(r.chain_id), str(r.agent_id)))
agent_docs = fetch_agents_by_keys(list(all_keys))
print('agents:', len(agent_docs))

def parse_services(raw_services: str) -> list[dict]:
    if not isinstance(raw_services, str) or not raw_services.strip():
        return []
    try:
        data = json.loads(raw_services)
    except json.JSONDecodeError:
        return []
    return data if isinstance(data, list) else []

def parse_tags(raw_tags: str) -> list[str]:
    if not isinstance(raw_tags, str) or not raw_tags.strip():
        return []
    return [x.strip() for x in raw_tags.replace('|', ',').split(',') if x.strip()]

def agent_meta(row):
    doc = agent_docs.get(f'{row.chain_id}:{row.agent_id}', {})
    csv_description = getattr(row, 'agent_description', '') or ''
    csv_services = getattr(row, 'agent_services', '') or ''
    csv_tags = getattr(row, 'agent_tags', '') or ''
    description = doc.get('description','') or csv_description
    return AgentMeta(
        chain_id=int(row.chain_id), agent_id=str(row.agent_id),
        name=doc.get('name','') or f'agent:{row.agent_id}',
        description=description,
        summary=doc.get('agentSummary','') or csv_description or description,
        services=doc.get('services', []) or parse_services(csv_services),
        oasf_domains=doc.get('oasfDomains', []) or [],
        oasf_skills=doc.get('oasfSkills', []) or [],
        tags=doc.get('tags', []) or parse_tags(csv_tags),
    )

def feedback_record(row):
    fp = row.feedback_parsed
    if isinstance(fp, str) and fp.strip().startswith('{'):
        try: fp = json.loads(fp)
        except json.JSONDecodeError: fp = None
    else: fp = None
    return FeedbackRecord(
        id=row.id, agent_id=str(row.agent_id), chain_id=int(row.chain_id),
        tag1=row.tag1 or '', tag2=row.tag2 or '',
        endpoint=row.endpoint or '', value=str(row.value),
        value_decimals=int(row.value_decimals), value_scale=row.value_scale or '',
        feedback_parsed=fp, rule_category=getattr(row, 'rule_category', row.label),
        is_self_feedback=bool(row.is_self_feedback),
    )

In [ ]:
TRAIN_ROWS = list(TRAIN.itertuples())
TEST_ROWS_BY_BENCH = {bench: list(df.itertuples()) for bench, df in TESTS.items()}

def few_shot_block(idxs: list[int]) -> str:
    parts = []
    for j in idxs:
        r = TRAIN_ROWS[j]
        fb = feedback_record(r); a = agent_meta(r)
        parts.append(build_user_message(a, fb) + f'\n=> {{"category":"{r.label}"}}')
    return 'EXAMPLES:\n' + '\n\n'.join(parts) + '\n\nNOW CLASSIFY:'

# Pre-build prompts for all test records in each benchmark.
prompts_by_bench = {}
for bench, rows in TEST_ROWS_BY_BENCH.items():
    prompts = []
    topk_idx = topk_by_bench[bench]
    for i, r in enumerate(tqdm(rows, desc=f'build prompts:{bench}')):
        fb = feedback_record(r); a = agent_meta(r)
        user = few_shot_block(topk_idx[i].tolist()) + '\n\n' + build_user_message(a, fb)
        prompts.append((r.id, r.label, user))
    prompts_by_bench[bench] = prompts
    print(f'{bench:13s} prompts:', len(prompts))
print('sample user msg (head):\n', prompts_by_bench['rule_based'][0][2][:600], '…')

In [ ]:
# Sweep a small panel — 3B (the hypothesis); 7B as a reasonable upper bound.
MODELS = ['qwen2.5:3b', 'qwen2.5:7b-instruct']
RESULTS_DIR = AI_ROOT / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

all_runs = []
for bench, prompts in prompts_by_bench.items():
    for m in MODELS:
        print(f'\n=== {bench} / {m} ===')
        out_path = RESULTS_DIR / f'approach_c_{bench}_{m.replace(":","_").replace("/","_")}.parquet'
        if out_path.exists():
            all_runs.append(pd.read_parquet(out_path))
            print('  cached, skipping')
            continue
        rows = []
        with OllamaClient(model=m, system_prompt=SYSTEM_V4) as cli:
            if not cli.health():
                print('  ⚠ Ollama unreachable')
                continue
            for fid, label, user in tqdm(prompts, desc=bench):
                res = cli.classify(user)
                rows.append({
                    'id': fid, 'bench': bench, 'model': f'C:{m}', 'gold': label, 'pred': res.category,
                    'confidence': res.confidence, 'reason': res.reason,
                    'latency_ms': res.latency_ms, 'source': res.source,
                })
        run_df = pd.DataFrame(rows)
        run_df.to_parquet(out_path, index=False)
        all_runs.append(run_df)
        print('  wrote', out_path)

In [ ]:
summary_rows = []
for run in all_runs:
    bench = run['bench'].iloc[0]
    m = run['model'].iloc[0]
    row = summarize_run(f'{bench}:{m}', run['gold'].tolist(), run['pred'].tolist(), run['latency_ms'].tolist())
    row['bench'] = bench
    row['model'] = m
    summary_rows.append(row)
pd.DataFrame(summary_rows).sort_values(['bench', 'macro_f1'], ascending=[True, False])

In [ ]:
if all_runs:
    combined = pd.concat(all_runs, ignore_index=True)
    combined.to_parquet(RESULTS_DIR / 'approach_c_all.parquet', index=False)
    print('combined →', RESULTS_DIR / 'approach_c_all.parquet', len(combined))